In [1]:

from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import networkx as nx

# ------------------------------------------------------------
# Configuración uniforme para las tres figuras
# ------------------------------------------------------------
SALIDA = Path('imagenes')  # Cambiar esta ruta si se desea.
SALIDA.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'pdf.fonttype': 42,       # texto vectorial incrustado como TrueType
    'ps.fonttype': 42,
    'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
})

# Paleta contrastada, sobria y coherente entre figuras.
COLOR = {
    'autor': '#DCECF6',
    'autor_borde': '#1476A2',
    'articulo': '#F9E9C7',
    'articulo_borde': '#B27A20',
    'keyword': '#DAF1E5',
    'keyword_borde': '#217E61',
    'afiliacion': '#F4DFEF',
    'afiliacion_borde': '#98527B',
    'linea': '#B3BFC7',
    'linea_oscura': '#6E8595',
    'texto': '#1E3442',
    'gris': '#E9EEF1',
    'gris_borde': '#A7B4BE',
    'camino_bfs': '#087CA3',
    'camino_dfs': '#BF6636',
    'grupo': '#D08728',
    'central': '#245673',
}


def guardar(fig, nombre):
    """Genera un PDF vectorial y un PNG a 350 DPI."""
    fig.savefig(SALIDA / f'{nombre}.pdf', bbox_inches='tight', pad_inches=0.13)
    fig.savefig(SALIDA / f'{nombre}.png', dpi=350,
                bbox_inches='tight', pad_inches=0.13)
    plt.close(fig)
    print(f'Guardada: {SALIDA / nombre}.pdf / .png')


def limpiar_ejes(ax, xlim, ylim):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_axis_off()

In [2]:
# FIGURA 1. Red bibliográfica y proyección de coautoría
# ============================================================
def figura_bibliografico_coautoria():
    """Las aristas autor-autor se DEDUCEN de artículos compartidos."""
    B = nx.Graph()
    nodos_por_tipo = {
        'autor': ['A1', 'A2', 'A3', 'A4'],
        'articulo': ['P1', 'P2', 'P3'],
        'keyword': ['K1', 'K2', 'K3', 'K4'],
        'afiliacion': ['F1', 'F2', 'F3'],
    }
    for tipo, nombres in nodos_por_tipo.items():
        B.add_nodes_from(nombres, tipo=tipo)

    # Autor-artículo. P1 y P2 son publicaciones DISTINTAS.
    autores_de_articulo = {
        'P1': ['A1', 'A2'],
        'P2': ['A2', 'A3'],
        'P3': ['A3', 'A4'],
    }
    # Artículo-palabra clave. A1 y A3 comparten la palabra K2 de
    # manera indirecta, pero NO una publicación en común.
    palabras_de_articulo = {
        'P1': ['K1', 'K2'],
        'P2': ['K2', 'K3'],
        'P3': ['K4'],
    }
    # Autor-afiliación (como nodos en la red principal).
    afiliacion_de_autor = {
        'A1': 'F1', 'A2': 'F1', 'A3': 'F2', 'A4': 'F3'
    }

    for articulo, autores in autores_de_articulo.items():
        B.add_edges_from((autor, articulo) for autor in autores)
    for articulo, palabras in palabras_de_articulo.items():
        B.add_edges_from((articulo, kw) for kw in palabras)
    for autor, afiliacion in afiliacion_de_autor.items():
        B.add_edge(autor, afiliacion)

    # Proyección de coautoría: SOLO nodos autor y aristas por
    # participación conjunta en un MISMO artículo.
    C = nx.Graph()
    C.add_nodes_from(nodos_por_tipo['autor'])
    for autores in autores_de_articulo.values():
        C.add_edges_from(combinations(sorted(set(autores)), 2))
    nx.set_node_attributes(C, afiliacion_de_autor, name='afiliacion')

    assert set(map(frozenset, C.edges())) == {
        frozenset(('A1', 'A2')),
        frozenset(('A2', 'A3')),
        frozenset(('A3', 'A4')),
    }
    assert not C.has_edge('A1', 'A3')

    fig = plt.figure(figsize=(13.4, 5.45))
    ax1 = fig.add_axes([0.035, 0.19, 0.623, 0.69])
    ax2 = fig.add_axes([0.708, 0.19, 0.255, 0.69])
    fig.text(0.34, 0.932, '(a) Grafo bibliográfico', ha='center',
             color=COLOR['texto'], fontsize=17, fontweight='semibold')
    fig.text(0.836, 0.932, '(b) Red de coautoría', ha='center',
             color=COLOR['texto'], fontsize=17, fontweight='semibold')

    # Posiciones fijas para garantizar reproducibilidad.
    pos = {
        'F1': (-4.8, 1.55), 'F2': (-4.8, -0.85), 'F3': (-4.8, -2.02),
        'A1': (-2.85, 2.12), 'A2': (-2.85, 0.57),
        'A3': (-2.85, -0.82), 'A4': (-2.85, -2.15),
        'P1': (-0.50, 1.52), 'P2': (-0.50, -0.17), 'P3': (-0.50, -1.92),
        'K1': (1.56, 2.27), 'K2': (1.56, 0.88),
        'K3': (1.56, -0.48), 'K4': (1.56, -1.96),
    }
    nx.draw_networkx_edges(B, pos, ax=ax1, width=2.0,
                           edge_color=COLOR['linea'], alpha=0.92)
    formas = {
        'autor': 'o', 'articulo': 's', 'keyword': 'D', 'afiliacion': 'h'
    }
    for tipo, nodos in nodos_por_tipo.items():
        nx.draw_networkx_nodes(
            B, pos, nodelist=nodos, ax=ax1, node_shape=formas[tipo],
            node_size=760 if tipo in {'autor', 'articulo'} else 690,
            node_color=COLOR[tipo], edgecolors=COLOR[tipo + '_borde'],
            linewidths=2.0
        )
    nx.draw_networkx_labels(B, pos, ax=ax1, font_size=13.5,
                            font_weight='semibold', font_color=COLOR['texto'])
    limpiar_ejes(ax1, (-5.48, 2.45), (-2.79, 2.83))

    # El grafo de coautoría almacena afiliaciones como atributos,
    # en vez de representar afiliaciones mediante nodos adicionales.
    pos_c = {
        'A1': (0.45, 2.02), 'A2': (2.2, 0.68),
        'A3': (0.45, -0.78), 'A4': (2.2, -2.12),
    }
    nx.draw_networkx_edges(C, pos_c, ax=ax2, width=3.1,
                           edge_color=COLOR['linea_oscura'])
    nx.draw_networkx_nodes(C, pos_c, ax=ax2, node_shape='o',
                           node_size=1280, node_color=COLOR['autor'],
                           edgecolors=COLOR['autor_borde'], linewidths=2.5)
    nx.draw_networkx_labels(C, pos_c, ax=ax2, font_size=15,
                            font_weight='semibold', font_color=COLOR['texto'])
    for autor, (x, y) in pos_c.items():
        ax2.annotate(
            f"{afiliacion_de_autor[autor]} (atributo)",
            (x, y), xytext=(0, -29), textcoords='offset points',
            ha='center', va='center', fontsize=10, color='#536875'
        )
    limpiar_ejes(ax2, (-0.36, 3.08), (-2.9, 2.75))

    leyenda = [
        Line2D([], [], ls='none', marker='o', ms=12, mec=COLOR['autor_borde'],
               mew=1.7, mfc=COLOR['autor'], label='Autor'),
        Line2D([], [], ls='none', marker='s', ms=12, mec=COLOR['articulo_borde'],
               mew=1.7, mfc=COLOR['articulo'], label='Artículo'),
        Line2D([], [], ls='none', marker='D', ms=11, mec=COLOR['keyword_borde'],
               mew=1.7, mfc=COLOR['keyword'], label='Palabra clave'),
        Line2D([], [], ls='none', marker='h', ms=12, mec=COLOR['afiliacion_borde'],
               mew=1.7, mfc=COLOR['afiliacion'], label='Afiliación'),
    ]
    fig.legend(handles=leyenda, loc='lower center',
               bbox_to_anchor=(0.5, 0.105), ncol=4, frameon=False,
               fontsize=11.5, columnspacing=2.25, handletextpad=0.65)
    fig.text(0.5, 0.027,
             'Ejemplo sintético: A1 y A3 comparten la palabra clave K2, pero no una publicación.',
             ha='center', va='center', fontsize=10.4, color='#566A76')
    guardar(fig, 'fig_grafos_01_bibliografico_coautoria')

figura_bibliografico_coautoria()

Guardada: imagenes\fig_grafos_01_bibliografico_coautoria.pdf / .png


In [3]:
# FIGURA 2. Grado, vecindad y agrupamiento local
# ============================================================
def figura_metricas_topologicas():
    G = nx.Graph()
    G.add_edges_from([
        ('C', 'A'), ('C', 'B'), ('C', 'D'), ('C', 'E'),
        ('A', 'B'), ('D', 'E'),
        ('B', 'F'), ('E', 'G'),
    ])
    vecinos = set(G.neighbors('C'))
    aristas_entre_vecinos = [
        (u, v) for u, v in G.edges() if u in vecinos and v in vecinos
    ]
    posibles = len(vecinos) * (len(vecinos) - 1) // 2
    assert vecinos == {'A', 'B', 'D', 'E'}
    assert len(aristas_entre_vecinos) == 2 and posibles == 6
    assert abs(nx.clustering(G, 'C') - (1 / 3)) < 1e-12

    pos = {
        'C': (0.00, 0.00),
        'A': (-1.18, 1.50), 'B': (1.18, 1.50),
        'D': (-1.18, -1.50), 'E': (1.18, -1.50),
        'F': (2.66, 2.02), 'G': (2.66, -2.02),
    }
    fig = plt.figure(figsize=(10.25, 5.22))
    ax = fig.add_axes([0.032, 0.14, 0.61, 0.75])
    panel = fig.add_axes([0.68, 0.19, 0.29, 0.68])

    # Enlaces ordinarios y los que intervienen en el cálculo.
    enlaces_al_centro = [(u, v) for u, v in G.edges() if 'C' in (u, v)]
    enlaces_externos = [(u, v) for u, v in G.edges()
                        if (u, v) not in enlaces_al_centro
                        and (u, v) not in aristas_entre_vecinos]
    nx.draw_networkx_edges(G, pos, edgelist=enlaces_externos, ax=ax,
                           edge_color=COLOR['linea'], width=2.1)
    nx.draw_networkx_edges(G, pos, edgelist=enlaces_al_centro, ax=ax,
                           edge_color=COLOR['camino_bfs'], width=2.8)
    nx.draw_networkx_edges(G, pos, edgelist=aristas_entre_vecinos, ax=ax,
                           edge_color=COLOR['grupo'], width=4.2)

    nx.draw_networkx_nodes(G, pos, nodelist=['F', 'G'], ax=ax,
                           node_size=830, node_color=COLOR['gris'],
                           edgecolors=COLOR['gris_borde'], linewidths=2)
    nx.draw_networkx_nodes(G, pos, nodelist=sorted(vecinos), ax=ax,
                           node_size=930, node_color=COLOR['autor'],
                           edgecolors=COLOR['autor_borde'], linewidths=2.1)
    nx.draw_networkx_nodes(G, pos, nodelist=['C'], ax=ax,
                           node_size=1150, node_color=COLOR['central'],
                           edgecolors='#17394A', linewidths=2.2)
    nx.draw_networkx_labels(G, pos, ax=ax,
                            labels={v: v for v in ['A', 'B', 'D', 'E', 'F', 'G']},
                            font_size=16, font_weight='semibold',
                            font_color=COLOR['texto'])
    nx.draw_networkx_labels(G, pos, ax=ax, labels={'C': 'C'},
                            font_size=16, font_weight='bold', font_color='white')
    limpiar_ejes(ax, (-2.10, 3.35), (-2.58, 2.66))

    panel.set_axis_off()
    panel.text(0.0, 0.98, 'Agrupamiento del nodo C',
               transform=panel.transAxes, ha='left', va='top',
               fontsize=15, fontweight='semibold', color=COLOR['texto'])
    entradas = [
        (r'$\Gamma(C)=\{A,B,D,E\}$', 0.80),
        (r'$k_C=4$', 0.65),
        ('Enlaces entre vecinos: 2', 0.50),
        ('Pares posibles: 6', 0.36),
    ]
    for texto, y in entradas:
        panel.text(0, y, texto, transform=panel.transAxes,
                   ha='left', va='center', fontsize=13.2, color=COLOR['texto'])
    panel.plot([0, 0.98], [0.245, 0.245], transform=panel.transAxes,
               color='#D3DADE', lw=1.4)
    panel.text(0.0, 0.12, r'$C_C = \frac{2}{6}=\frac{1}{3}$',
               transform=panel.transAxes, ha='left', va='center',
               fontsize=21, fontweight='bold', color=COLOR['central'])

    leyenda = [
        Line2D([], [], ls='none', marker='o', ms=10,
               mfc=COLOR['central'], mec=COLOR['central'], label='Nodo C'),
        Line2D([], [], ls='none', marker='o', ms=10,
               mfc=COLOR['autor'], mec=COLOR['autor_borde'], label='Vecinos de C'),
        Line2D([], [], lw=3.5, color=COLOR['grupo'],
               label='Aristas entre vecinos'),
        Line2D([], [], ls='none', marker='o', ms=10,
               mfc=COLOR['gris'], mec=COLOR['gris_borde'], label='Otros nodos'),
    ]
    fig.legend(handles=leyenda, loc='lower center', bbox_to_anchor=(0.5, 0.032),
               ncol=4, fontsize=10.5, frameon=False,
               columnspacing=1.35, handletextpad=0.6)
    guardar(fig, 'fig_grafos_02_metricas_topologicas')

figura_metricas_topologicas()

Guardada: imagenes\fig_grafos_02_metricas_topologicas.pdf / .png


In [4]:
# FIGURA 3. BFS vs. DFS y longitud de camino en grafo no ponderado
# ============================================================
def primer_camino_dfs(G, inicio, destino):
    """Primer camino a destino mediante DFS, con vecinos en orden alfabético."""
    visitados = set()

    def explorar(actual, camino):
        visitados.add(actual)
        if actual == destino:
            return camino
        for vecino in sorted(G.neighbors(actual)):
            if vecino not in visitados:
                resultado = explorar(vecino, camino + [vecino])
                if resultado is not None:
                    return resultado
        return None

    return explorar(inicio, [inicio])


def figura_busquedas_caminos():
    G = nx.Graph()
    G.add_edges_from([
        ('A', 'B'), ('A', 'C'), ('A', 'G'),
        ('B', 'C'), ('C', 'E'), ('C', 'D'),
        ('D', 'E'), ('E', 'F'), ('F', 'G'),
    ])
    inicio, destino = 'A', 'F'
    camino_bfs = nx.shortest_path(G, inicio, destino)
    camino_dfs = primer_camino_dfs(G, inicio, destino)
    niveles = nx.single_source_shortest_path_length(G, inicio)

    assert camino_bfs == ['A', 'G', 'F']
    assert camino_dfs == ['A', 'B', 'C', 'D', 'E', 'F']
    assert len(camino_bfs) - 1 == 2 and len(camino_dfs) - 1 == 5
    assert {n for n, d in niveles.items() if d == 1} == {'B', 'C', 'G'}
    assert {n for n, d in niveles.items() if d == 2} == {'D', 'E', 'F'}

    pos = {
        'A': (-2.10, 0.00),
        'B': (-1.20, 1.25), 'C': (-0.05, 1.10),
        'D': (1.16, 1.28), 'E': (1.30, -0.18),
        'F': (2.70, -0.46), 'G': (-0.10, -1.16),
    }

    fig, ejes = plt.subplots(1, 2, figsize=(12.5, 5.15))
    fig.subplots_adjust(left=0.035, right=0.965,
                        top=0.81, bottom=0.19, wspace=0.075)

    for ax, recorrido, color, titulo, descripcion in (
        (ejes[0], camino_bfs, COLOR['camino_bfs'],
         '(a) BFS: búsqueda en anchura', 'Camino mínimo: A - G - F (2 aristas)'),
        (ejes[1], camino_dfs, COLOR['camino_dfs'],
         '(b) DFS: búsqueda en profundidad', 'Primer camino: A - B - C - D - E - F (5 aristas)'),
    ):
        limpiar_ejes(ax, (-2.72, 3.15), (-1.61, 1.76))
        # Se mantiene la MISMA red y las MISMAS posiciones en ambos paneles.
        nx.draw_networkx_edges(G, pos, ax=ax,
                               edge_color=COLOR['linea'], width=2.1)
        enlaces_recorrido = list(zip(recorrido[:-1], recorrido[1:]))
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=enlaces_recorrido,
                               edge_color=color, width=4.3)
        otros = [v for v in G.nodes() if v not in recorrido]
        intermedios = [v for v in recorrido if v not in (inicio, destino)]
        nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=otros,
                               node_color=COLOR['gris'], edgecolors=COLOR['gris_borde'],
                               node_size=990, linewidths=1.9)
        nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=intermedios,
                               node_color=COLOR['autor'], edgecolors=color,
                               node_size=1060, linewidths=2.8)
        nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=[inicio, destino],
                               node_color=COLOR['central'], edgecolors=COLOR['central'],
                               node_size=1170, linewidths=2.2)
        nx.draw_networkx_labels(
            G, pos, labels={u: u for u in G.nodes() if u not in (inicio, destino)},
            ax=ax, font_size=16, font_weight='semibold', font_color=COLOR['texto']
        )
        nx.draw_networkx_labels(
            G, pos, labels={inicio: inicio, destino: destino}, ax=ax,
            font_size=16, font_weight='bold', font_color='white'
        )
        ax.set_title(titulo, fontsize=15, fontweight='semibold',
                     color=COLOR['texto'], pad=21)
        ax.text(0.50, -0.095, descripcion, transform=ax.transAxes,
                ha='center', va='top', color=color,
                fontsize=12.8, fontweight='semibold')

    fig.text(0.5, 0.044,
             'Grafo no ponderado; en DFS se visitan los vecinos en orden alfabético.',
             fontsize=11.2, ha='center', color='#566A76')
    guardar(fig, 'fig_grafos_03_busquedas_caminos')

figura_busquedas_caminos()

Guardada: imagenes\fig_grafos_03_busquedas_caminos.pdf / .png
